# NB17 — CFTR Panel Refinement: 7 Strateji Karşılaştırması

**TEKNOFEST Sağlıkta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

## Problem

CFTR panelinin **temel sorunu model değil, veri:**
- n=21 benign — NB15/NB16'da bootstrap CI=[0.00–1.00] → hiçbir model karşılaştırması anlamlı değil
- Eğitim dağılımı: %81 pathogenic / %19 benign
- Final test dağılımı: ~%80 benign / %20 pathogenic (tam tersi!)

## Üç Katmanlı Çözüm

1. **Değerlendirme sorunu → LOO-CV + MCC:** Tüm 111 örnek değerlendirmede, n=21 benign kullanılır.  
   Birincil metrik: LOO-CV MCC (kararlı, CI hesaplanabilir).  
   Referans: Chicco & Jurman (2020) BMC Genomics — küçük n'de MCC > F1.

2. **Dağılım kayması → Prior Shift düzeltmesi:** Model olasılıklarını π_train=0.73 → π_test=0.20'ye kalibre et.  
   Formül: `p_adj = (p*R) / (p*R + 1-p)` — R = (π_test/(1-π_test)) / (π_train/(1-π_train))  
   Referans: Saerens et al. (2002) Neural Computation.  

3. **Modelleme sorunu → 7 strateji tek çerçevede:**

| ID | Strateji | Eğitim | Özel |
|----|----------|---------|------|
| S0 | MASTER-only | Tüm MASTER (2931) | Kullanıcı önerisi — baseline |
| S1 | MASTER-core | 625/625 dengeli çekirdek | NB16 S1 mirror |
| S2 | SMOTE-CFTR | CFTR + SMOTE (benign 21→40) | imblearn, k=3 |
| S3 | TabPFN | CFTR tüm, top-100 SHAP | Yüklü değilse atlanır |
| S4 | FrozenFinetune | MASTER pretrain + frozen body | Yalnızca son katman fine-tune |
| S5 | BalancedBagging | MASTER (dengeli bagging) | imblearn.ensemble |
| S6 | PriorShift | S0 çıktısı + post-hoc kalibrasyon | Yeniden eğitim yok |

## Değerlendirme Protokolü

1. **LOO-CV MCC** — BİRİNCİL (tüm 111 örnek kullanılır, kararlı)
2. **Bootstrap %80/20 F1** — NB16 kıyası için (N=50)
3. **Multi-seed 50/50** — 20 farklı seed, tek-split yanılsamasını kırar

## M3 Missing + FE

NB16 ile aynı: `fit_preprocessor()` MASTER üzerinde fit, CFTR'ye transform. FE: Grantham/BLOSUM62/stopgain.

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split, LeaveOneOut, cross_val_predict
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             matthews_corrcoef, confusion_matrix)
from sklearn.pipeline import Pipeline
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
from src import columns_real as CR
from src.models import LGBM_FIXED, CB_FIXED
from src.metrics import optimize_threshold
from src.focal_loss import FocalLoss
import lightgbm as lgb
from catboost import CatBoostClassifier

# SMOTE / BalancedBagging -- graceful import
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.ensemble import BalancedBaggingClassifier
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("[UYARI] imbalanced-learn yuklu degil -- S2 ve S5 atlanacak")

# TabPFN -- graceful import
try:
    from tabpfn import TabPFNClassifier
    HAS_TABPFN = True
except ImportError:
    HAS_TABPFN = False
    print("[UYARI] tabpfn yuklu degil -- S3 atlanacak (pip install tabpfn ile ekle)")

np.random.seed(SEED); torch.manual_seed(SEED)
try:
    import torch_directml; DEVICE = torch_directml.device()
except Exception:
    DEVICE = torch.device("cpu")

# --- Sabitler ---
PANEL             = "CFTR"
MASTER_CORE_POS   = MASTER_CORE_NEG = 625  # NB15/16 ile tutarli
HIGH_MISSING_THR  = 0.50
FINAL_BENIGN_FRAC = 0.80
N_BOOT            = 50
BOOT_SEED         = SEED
N_MULTISEED       = 20   # multi-seed 50/50 degerlendirme icin
PANEL_SPLIT_FRAC  = 0.50
PI_TRAIN          = 0.73  # MASTER label=1 orani (yaklasik)
PI_TEST           = 0.20  # yarisma final test orani
TABPFN_MAX_FEAT   = 100   # TabPFN hard limit
SMOTE_K           = 3     # k_neighbors -- n_benign=21 icin kucuk
BAGGING_N         = 30

DATA_DIR    = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v7_cftr_refinement")
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}  DEVICE: {DEVICE}  SEED: {SEED}")
print(f"imblearn: {HAS_IMBLEARN}  tabpfn: {HAS_TABPFN}")

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model  DEVICE: cpu  SEED: 42
imblearn: True  tabpfn: True


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi (NB16 ile ayni mantik)
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

master = load_panel("MASTER")
cftr_raw = load_panel("CFTR")
feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

# Cross-panel birebir-ayni satir drop (CFTR icin beklenen: 0)
master_index = {}
for _, row in master.iterrows():
    key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v) for v in row[feature_cols_all].values)
    master_index[key] = row[TARGET]
drop_idx = [idx for idx, row in cftr_raw.iterrows()
            if (((row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                 for v in row[feature_cols_all].values)) in master_index
                and master_index[(row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                     for v in row[feature_cols_all].values)] == row[TARGET])]
cftr = cftr_raw.drop(index=drop_idx).reset_index(drop=True)
print(f"MASTER: {master.shape}, label: {master[TARGET].value_counts().to_dict()}")
print(f"CFTR  : {cftr.shape}, label: {cftr[TARGET].value_counts().to_dict()} (drop={len(drop_idx)})")

# Sutun temizligi: sabit + ozdes -- MASTER uzerinde (NB15/16 ile ayni 63 sutun)
constant_cols = CR.get_constant_cols(master[feature_cols_all])
dup_pairs     = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop      = sorted({b for (a, b) in dup_pairs})
drop_cols     = sorted(set(constant_cols) | set(dup_drop))
base_feature_cols = [c for c in feature_cols_all if c not in drop_cols]
CAT_LIKE      = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in base_feature_cols]
NUM_COLS_BASE = [c for c in base_feature_cols if c not in CAT_LIKE]

print(f"drop {len(drop_cols)} sutun -> {len(base_feature_cols)} ham feature "
      f"({len(NUM_COLS_BASE)} sayisal + {len(CAT_LIKE)} kategorik)")

MASTER: (2931, 353), label: {1: 2149, 0: 782}
CFTR  : (111, 353), label: {1: 90, 0: 21} (drop=0)
drop 63 sutun -> 288 ham feature (281 sayisal + 7 kategorik)


In [3]:
# Cell 3: FE Fonksiyonlari + M3 Preprocessing (NB16 Cell 4+5'ten aynen alinmis)
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# --- Grantham (1974) mesafe matrisi ---
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or -1

# --- BLOSUM62 ---
_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for _ri, _line in enumerate(_B62_RAW.strip().split("\n")):
    _toks = _line.split()
    _row_aa = _toks[0][0]
    _vals = [_toks[0][1:]] + _toks[1:]
    for _ci, _tok in enumerate(_vals):
        _col_aa = _ORDER[_ri + _ci]
        _v = int(_tok[1:] if _tok[0].isalpha() else _tok)
        _B62[(_row_aa, _col_aa)] = _v; _B62[(_col_aa, _row_aa)] = _v
def blosum62(a, b): return _B62.get((a, b), 0)

def add_fe(df):
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v): return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]

def detect_log_cols(train_df):
    cand = []
    for c in NUM_COLS_BASE:
        s = pd.to_numeric(train_df[c], errors="coerce").dropna()
        if len(s) < 10: continue
        if s.min() >= 0 and s.max() > 1.0 and s.skew() > 2.0:
            cand.append(c)
    return cand

def fit_preprocessor(train_df_raw):
    tr = add_fe(train_df_raw)
    log_cols = detect_log_cols(tr)
    num_cols = NUM_COLS_BASE + FE_NEW_COLS + [f"{c}__log" for c in log_cols]
    for c in log_cols:
        tr[f"{c}__log"] = np.log1p(pd.to_numeric(tr[c], errors="coerce").clip(lower=0))
    miss = tr[base_feature_cols].isna().mean()
    flag_source = miss[miss > HIGH_MISSING_THR].index.tolist()
    median = {c: pd.to_numeric(tr[c], errors="coerce").median() for c in num_cols}
    return {"log_cols": log_cols, "num_cols": num_cols, "cat_cols": CAT_LIKE,
            "flag_source": flag_source, "median": median}

def transform_X(df_raw, pp):
    df = add_fe(df_raw)
    for c in pp["log_cols"]:
        df[f"{c}__log"] = np.log1p(pd.to_numeric(df[c], errors="coerce").clip(lower=0))
    out = pd.DataFrame(index=df.index)
    for c in pp["num_cols"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(pp["median"][c]).astype(float).values
    for c in pp["cat_cols"]:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = df[c].astype("object").where(~df[c].isna(), fill).astype(str).values
    for c in pp["flag_source"]:
        out[CR.get_missing_mask_col_name(c)] = df[c].isna().astype(int).values
    return out, list(pp["cat_cols"])

# MASTER'da fit
pp = fit_preprocessor(master)
X_master_df, cat_cols = transform_X(master, pp)
y_master = master[TARGET].values
X_cftr_df, _ = transform_X(cftr, pp)
y_cftr = cftr[TARGET].values

print(f"FE+M3 sonrasi: {X_master_df.shape[1]} sutun (log:{len(pp['log_cols'])}, "
      f"flag:{len(pp['flag_source'])}, cat:{len(cat_cols)}) NaN:{X_master_df.isna().any().any()}")
print(f"CFTR X: {X_cftr_df.shape}  y: {np.bincount(y_cftr.astype(int))}")

FE+M3 sonrasi: 432 sutun (log:0, flag:140, cat:7) NaN:False
CFTR X: (111, 432)  y: [21 90]


In [4]:
# Cell 4: Degerlendirme Altyapisi -- LOO-CV, bootstrap %80/20, multi-seed, prior shift

# --- 4a. Prior shift duzeltmesi (Saerens et al. 2002) ---
def adjust_prior_shift(proba, pi_train=PI_TRAIN, pi_test=PI_TEST):
    """Model olasiligini pi_train -> pi_test icin kalibre et."""
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# --- 4b. Bootstrap %80/20 (NB16 ile ayni) ---
def _f1_pos(y, p): return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0: return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED); f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020(y, prob):
    """Tek 8020 yeniden orneklemede F1-max threshold sec."""
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best: best, best_thr = f, thr
    return float(best_thr)

# --- 4c. LOO-CV degerlendirme (tum 111 ornek kullanilir) ---
def loo_metrics(y_true, oof_proba, prior_shift=False):
    """LOO-CV olasiliklarindan MCC, F1, AUC-ROC hesapla.
    prior_shift=True ise adjust_prior_shift uygular."""
    prob = adjust_prior_shift(oof_proba) if prior_shift else oof_proba
    thr = select_threshold_8020(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1  = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    return {"mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "y_pred": y_pred, "y_true": y_true, "prob": prob}

# --- 4d. Multi-seed 50/50 degerlendirme ---
def multiseed_eval(fit_fn, n_seeds=N_MULTISEED):
    """n_seeds farkli 50/50 split uzerinde fit_fn(X_tr,y_tr,X_te,y_te)->proba cagir.
    fit_fn: (X_train_df, y_train, X_test_df, y_test) -> (proba_train, proba_test)
    Dondurulen: f1 ortalama, std, mcc ortalama."""
    f1s, mccs = [], []
    for seed in range(n_seeds):
        pos_idx = np.where(y_cftr == 1)[0]
        neg_idx = np.where(y_cftr == 0)[0]
        rng = np.random.RandomState(seed)
        rng.shuffle(pos_idx); rng.shuffle(neg_idx)
        nhalf_pos = len(pos_idx) // 2; nhalf_neg = len(neg_idx) // 2
        tr_idx = np.concatenate([pos_idx[:nhalf_pos], neg_idx[:nhalf_neg]])
        te_idx = np.concatenate([pos_idx[nhalf_pos:], neg_idx[nhalf_neg:]])
        X_tr = X_cftr_df.iloc[tr_idx].reset_index(drop=True)
        y_tr = y_cftr[tr_idx]
        X_te = X_cftr_df.iloc[te_idx].reset_index(drop=True)
        y_te = y_cftr[te_idx]
        try:
            p_tr, p_te = fit_fn(X_tr, y_tr, X_te, y_te)
            thr = select_threshold_8020(y_tr, p_tr)
            yp = (p_te >= thr).astype(int)
            f1s.append(_f1_pos(y_te, yp))
            mccs.append(matthews_corrcoef(y_te, yp) if len(np.unique(y_te)) > 1 else 0.0)
        except Exception:
            pass
    return {"f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)),
            "mcc_mean": float(np.mean(mccs))}

# --- Train metrigi (overfit kontrolu icin) ---
def train_metrics_at(y_tr, p_tr, thr):
    yp = (p_tr >= thr).astype(int)
    return {
        "train_f1": _f1_pos(y_tr, yp),
        "train_mcc": matthews_corrcoef(y_tr, yp) if len(np.unique(y_tr)) > 1 else 0.0,
        "train_precision": precision_score(y_tr, yp, pos_label=1, zero_division=0),
        "train_recall": recall_score(y_tr, yp, pos_label=1, zero_division=0),
    }

print("Degerlendirme altyapisi hazir (LOO-MCC, bootstrap %80/20, multi-seed, prior-shift).")

Degerlendirme altyapisi hazir (LOO-MCC, bootstrap %80/20, multi-seed, prior-shift).


In [5]:
# Cell 5: Model Yardimcilari (NB16 Cell 7'den uyarlandi)

# --- LightGBM (kategorik label-encode ile, LOO-CV uyumlu) ---
def _le_encode_for_loo(X_df):
    """Kategorik sutunlari label-encode et (LOO icin categorical_feature verilemez)."""
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c].astype(str))
        le_maps[c] = le
    return Xn.astype(float), le_maps

def _lgbm_classifier():
    return lgb.LGBMClassifier(**{**LGBM_FIXED, "n_estimators": 200,
                                  "num_leaves": 31, "learning_rate": 0.05})

def _cb_classifier():
    return CatBoostClassifier(**{**CB_FIXED, "iterations": 200,
                                  "depth": 4, "learning_rate": 0.05})

# --- SmallMLP (NB16 ile ayni) ---
class SmallMLP(torch.nn.Module):
    def __init__(self, d, hidden, n_layers, dropout):
        super().__init__(); layers = []; inp = d
        for _ in range(n_layers):
            layers += [torch.nn.Linear(inp, hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout)]
            inp = hidden
        layers += [torch.nn.Linear(inp, 1)]; self.net = torch.nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

def _nn_encode_df(X_df):
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        le = LabelEncoder(); Xn[c] = le.fit_transform(Xn[c].astype(str)); le_maps[c] = le
    Xn = Xn.astype(np.float32)
    sc = StandardScaler().fit(Xn.values)
    return sc.transform(Xn.values), le_maps, sc, list(Xn.columns)

def _nn_apply(X_df, le_maps, sc, cols):
    Xn = X_df.copy()
    for c in cat_cols:
        Xn[c] = Xn[c].astype(str).map({cls: i for i, cls in enumerate(le_maps[c].classes_)}).fillna(-1)
    return sc.transform(Xn[cols].astype(np.float32).values)

def _train_es(model, Xt, yt, lr=1e-3, max_epochs=60, patience=10):
    """FocalLoss + early stopping (NB16 ile ayni)."""
    yv = np.asarray(yt)
    strat = yv if min((yv == 0).sum(), (yv == 1).sum()) >= 2 else None
    tri, vai = train_test_split(np.arange(len(yv)), test_size=0.25, random_state=SEED, stratify=strat)
    pw = torch.FloatTensor([(yv[tri] == 0).sum() / max((yv[tri] == 1).sum(), 1)])
    crit = FocalLoss(alpha=0.25, gamma=2.0, pos_weight=pw)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    Xtt = torch.FloatTensor(Xt[tri]); ytt = torch.FloatTensor(yv[tri].astype(np.float32))
    Xvl = torch.FloatTensor(Xt[vai]); yvl = yv[vai]
    n = len(Xtt); bs = min(64, max(2, n - 1)); best, state, pat = -1, None, 0
    for _ in range(max_epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            if len(idx) < 2: continue
            opt.zero_grad(); loss = crit(model(Xtt[idx]), ytt[idx]); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): vp = torch.sigmoid(model(Xvl)).numpy()
        f = _f1_pos(yvl, (vp >= 0.5).astype(int))
        if f > best: best, state, pat = f, deepcopy(model.state_dict()), 0
        else:
            pat += 1
            if pat >= patience: break
    if state is not None: model.load_state_dict(state)
    return model

def _nn_proba(model, X_np):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(torch.FloatTensor(X_np))).numpy()

# --- MASTER core (NB16 ile ayni) ---
def make_master_core(npos=MASTER_CORE_POS, nneg=MASTER_CORE_NEG):
    pos = master[master[TARGET] == 1].sample(n=min(npos, (master[TARGET] == 1).sum()), random_state=SEED)
    neg = master[master[TARGET] == 0].sample(n=min(nneg, (master[TARGET] == 0).sum()), random_state=SEED)
    return pd.concat([pos, neg]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

MASTER_CORE = make_master_core()
X_master_core_df, _ = transform_X(MASTER_CORE, pp)
y_master_core = MASTER_CORE[TARGET].values

print(f"MASTER CORE: {X_master_core_df.shape}  y: {np.bincount(y_master_core)}")
print("Model yardimcilari hazir.")

MASTER CORE: (1250, 432)  y: [625 625]
Model yardimcilari hazir.


In [6]:
# Cell 6: 7 Strateji -- LOO-CV + Multi-seed + Train Metrikleri
#
# Her strateji icin:
#   1. LOO-CV ile tum 111 CFTR uzerinde OOF olasilik
#   2. Multi-seed 50/50 gosterge degerlendirmesi
#   3. Train metrigi (overfit kontrolu)
#
# NOT: S2/S3/S5 icin LOO-CV calisma bicimi aciklanmistir (her fold icinde fit).

print("=== LOO-CV basliyor (111 fold x her strateji) ===")

all_results = {}

# --- Yardimci: LGBM label-encode edilmis X uzerinde LOO ---
X_cftr_le, le_maps_cftr = _le_encode_for_loo(X_cftr_df)
X_master_le, _           = _le_encode_for_loo(X_master_df)
X_master_core_le, _      = _le_encode_for_loo(X_master_core_df)

# --- S0: MASTER-only (tum MASTER ile egit, CFTR uzerinde LOO predict) ---
print("S0: MASTER-only...")
m_s0 = _lgbm_classifier(); m_s0.fit(X_master_le, y_master)
p_s0_train = m_s0.predict_proba(X_master_le)[:, 1]
# CFTR icin LOO: her fold'da MASTER ile ayni model kullanilir (MASTER degismez)
# -> cross_val_predict burada kullanilamaz (train CFTR degil, MASTER)
# -> direkt test olarak tum CFTR predict edilir (LOO degil, ama buyuk train seti)
p_s0_cftr = m_s0.predict_proba(X_cftr_le)[:, 1]
# Train metrigi MASTER uzerinde
thr_s0 = select_threshold_8020(y_master, p_s0_train)
train_m_s0 = train_metrics_at(y_master, p_s0_train, thr_s0)
loo_s0 = loo_metrics(y_cftr, p_s0_cftr)
# Multi-seed: her seed'de MASTER ile egit, CFTR testinde deger
def _s0_fit_fn(Xtr, ytr, Xte, yte):
    m = _lgbm_classifier(); m.fit(X_master_le, y_master)
    Xte_le, _ = _le_encode_for_loo(Xte)
    return m.predict_proba(X_master_le)[:, 1][:len(Xtr)], m.predict_proba(Xte_le)[:, 1]
ms_s0 = multiseed_eval(_s0_fit_fn)
all_results["S0_MASTER-only"] = {"loo": loo_s0, "ms": ms_s0, "train": train_m_s0, "prior_loo": loo_metrics(y_cftr, p_s0_cftr, prior_shift=True)}
print(f"  LOO MCC={loo_s0['mcc']:.4f}  F1={loo_s0['f1']:.4f}  AUC={loo_s0['auc']:.4f}  boot-mean={loo_s0['boot8020']['mean']:.4f}")

# --- S1: MASTER-core (625/625 dengeli) ---
print("S1: MASTER-core (625/625)...")
m_s1 = _lgbm_classifier(); m_s1.fit(X_master_core_le, y_master_core)
p_s1_train = m_s1.predict_proba(X_master_core_le)[:, 1]
p_s1_cftr  = m_s1.predict_proba(X_cftr_le)[:, 1]
thr_s1 = select_threshold_8020(y_master_core, p_s1_train)
train_m_s1 = train_metrics_at(y_master_core, p_s1_train, thr_s1)
loo_s1 = loo_metrics(y_cftr, p_s1_cftr)
def _s1_fit_fn(Xtr, ytr, Xte, yte):
    m = _lgbm_classifier(); m.fit(X_master_core_le, y_master_core)
    Xte_le, _ = _le_encode_for_loo(Xte)
    return p_s1_train[:len(Xtr)], m.predict_proba(Xte_le)[:, 1]
ms_s1 = multiseed_eval(_s1_fit_fn)
all_results["S1_MASTER-core"] = {"loo": loo_s1, "ms": ms_s1, "train": train_m_s1, "prior_loo": loo_metrics(y_cftr, p_s1_cftr, prior_shift=True)}
print(f"  LOO MCC={loo_s1['mcc']:.4f}  F1={loo_s1['f1']:.4f}  AUC={loo_s1['auc']:.4f}  boot-mean={loo_s1['boot8020']['mean']:.4f}")

# --- S2: SMOTE-CFTR ---
if HAS_IMBLEARN:
    print("S2: SMOTE-CFTR...")
    def _s2_loo_fn(Xtr_df, ytr, Xte_df, yte):
        Xtr_le, _ = _le_encode_for_loo(Xtr_df); Xte_le, _ = _le_encode_for_loo(Xte_df)
        # SMOTE yalnizca train uzerinde
        n_benign = (ytr == 0).sum()
        k = min(SMOTE_K, max(1, n_benign - 1))
        if n_benign >= 2:
            sm = SMOTE(k_neighbors=k, random_state=SEED)
            Xtr_le_np = Xtr_le.values if hasattr(Xtr_le, 'values') else np.array(Xtr_le)
            Xtr_aug, ytr_aug = sm.fit_resample(Xtr_le_np, ytr)
        else:
            Xtr_aug, ytr_aug = (Xtr_le.values if hasattr(Xtr_le, 'values') else np.array(Xtr_le)), ytr
        m = _lgbm_classifier(); m.fit(Xtr_aug, ytr_aug)
        Xte_le_np = Xte_le.values if hasattr(Xte_le, 'values') else np.array(Xte_le)
        return m.predict_proba(Xtr_aug)[:len(ytr), 1], m.predict_proba(Xte_le_np)[:, 1]
    # LOO: CFTR uzerinde her fold'da SMOTE uygula
    loo = LeaveOneOut()
    oof_s2 = np.zeros(len(y_cftr))
    p_s2_train_list = []
    for tri, vai in loo.split(X_cftr_le):
        Xtr = X_cftr_le.iloc[tri] if hasattr(X_cftr_le, 'iloc') else X_cftr_le[tri]
        ytr = y_cftr[tri]; Xte = X_cftr_le.iloc[vai] if hasattr(X_cftr_le, 'iloc') else X_cftr_le[vai]
        n_ben = (ytr == 0).sum(); k = min(SMOTE_K, max(1, n_ben - 1))
        Xtr_np = Xtr.values if hasattr(Xtr, 'values') else np.array(Xtr)
        Xte_np = Xte.values if hasattr(Xte, 'values') else np.array(Xte)
        if n_ben >= 2:
            sm = SMOTE(k_neighbors=k, random_state=SEED)
            Xtr_aug, ytr_aug = sm.fit_resample(Xtr_np, ytr)
        else:
            Xtr_aug, ytr_aug = Xtr_np, ytr
        m = _lgbm_classifier(); m.fit(Xtr_aug, ytr_aug)
        oof_s2[vai] = m.predict_proba(Xte_np)[:, 1]
        p_s2_train_list.append(m.predict_proba(Xtr_aug)[:len(ytr), 1].mean())
    loo_s2 = loo_metrics(y_cftr, oof_s2)
    ms_s2 = multiseed_eval(_s2_loo_fn)
    # Train metrigi: tum CFTR+SMOTE uzerinde model fit
    Xcftr_np = X_cftr_le.values if hasattr(X_cftr_le, 'values') else np.array(X_cftr_le)
    sm_full = SMOTE(k_neighbors=SMOTE_K, random_state=SEED)
    Xaug_full, yaug_full = sm_full.fit_resample(Xcftr_np, y_cftr)
    m_s2_full = _lgbm_classifier(); m_s2_full.fit(Xaug_full, yaug_full)
    p_s2_full_train = m_s2_full.predict_proba(Xaug_full)[:len(y_cftr), 1]
    thr_s2 = select_threshold_8020(y_cftr, p_s2_full_train)
    train_m_s2 = train_metrics_at(y_cftr, p_s2_full_train, thr_s2)
    all_results["S2_SMOTE-CFTR"] = {"loo": loo_s2, "ms": ms_s2, "train": train_m_s2, "prior_loo": loo_metrics(y_cftr, oof_s2, prior_shift=True)}
    print(f"  LOO MCC={loo_s2['mcc']:.4f}  F1={loo_s2['f1']:.4f}  AUC={loo_s2['auc']:.4f}  boot-mean={loo_s2['boot8020']['mean']:.4f}")
else:
    all_results["S2_SMOTE-CFTR"] = None; print("S2: ATLANDI (imblearn yok)")

# --- S3: TabPFN --- v8+ lisans + farkli API gerektiriyor, devre disi
# Etkinlestirmek icin: pip install tabpfn, lisans kabulu, TABPFN_TOKEN env var.
all_results["S3_TabPFN"] = None
print("S3: ATLANDI (TabPFN v8 lisans/API uyumsuzlugu -- etkinlestirmek icin README'e bak)")

# --- S4: FrozenFinetune (MASTER pretrain, son katman finetune) ---
print("S4: FrozenFinetune (SmallMLP, frozen body)...")
# Encoder: tum MASTER uzerinde fit
Xm_np, le_m, sc_m, cols_m = _nn_encode_df(X_master_df)
ym_np = y_master.astype(np.float32)
# Pretrain
base_nn = SmallMLP(Xm_np.shape[1], 128, 2, 0.4)
base_nn = _train_es(base_nn, Xm_np, ym_np, lr=1e-3, max_epochs=60, patience=10)
# LOO uzerinde: pretrain sabit (fold-bagimsiz), sadece son katman finetune
Xcftr_nn = _nn_apply(X_cftr_df, {c: le_m[c] for c in cat_cols}, sc_m, cols_m)
ycftr_np = y_cftr.astype(np.float32)
loo = LeaveOneOut()
oof_s4 = np.zeros(len(y_cftr))
for tri, vai in loo.split(Xcftr_nn):
    m = deepcopy(base_nn)
    # Frozen: sadece son lineer katman guncellenir
    for name, param in m.named_parameters():
        param.requires_grad = ("net." + str(len(list(m.net.children())) - 1) in name
                               or name.endswith("-1.weight") or name.endswith("-1.bias"))
    # Son lineer katmani bul (Sequential icindeki son modul)
    for name, param in m.named_parameters(): param.requires_grad = False
    last_layer = None
    for name, mod in m.named_modules():
        if isinstance(mod, torch.nn.Linear): last_layer = name
    for name, param in m.named_parameters():
        if last_layer and (name.startswith(last_layer)):
            param.requires_grad = True
    m = _train_es(m, Xcftr_nn[tri], ycftr_np[tri], lr=1e-4, max_epochs=30, patience=8)
    oof_s4[vai] = _nn_proba(m, Xcftr_nn[vai])
loo_s4 = loo_metrics(y_cftr, oof_s4)
# Train metrigi: pretrain + frozen finetune tum CFTR
m_s4_full = deepcopy(base_nn)
for name, param in m_s4_full.named_parameters(): param.requires_grad = False
last_layer = None
for name, mod in m_s4_full.named_modules():
    if isinstance(mod, torch.nn.Linear): last_layer = name
for name, param in m_s4_full.named_parameters():
    if last_layer and name.startswith(last_layer): param.requires_grad = True
m_s4_full = _train_es(m_s4_full, Xcftr_nn, ycftr_np, lr=1e-4, max_epochs=30, patience=8)
p_s4_train = _nn_proba(m_s4_full, Xcftr_nn)
thr_s4 = select_threshold_8020(y_cftr, p_s4_train)
train_m_s4 = train_metrics_at(y_cftr, p_s4_train, thr_s4)
# Multi-seed (basit: MASTER pretrain sabit, CFTR split'te finetune)
def _s4_fit_fn(Xtr_df, ytr, Xte_df, yte):
    Xtr_nn = _nn_apply(Xtr_df, {c: le_m[c] for c in cat_cols}, sc_m, cols_m)
    Xte_nn = _nn_apply(Xte_df, {c: le_m[c] for c in cat_cols}, sc_m, cols_m)
    m = deepcopy(base_nn)
    for name, param in m.named_parameters(): param.requires_grad = False
    for name, param in m.named_parameters():
        if last_layer and name.startswith(last_layer): param.requires_grad = True
    m = _train_es(m, Xtr_nn, ytr.astype(np.float32), lr=1e-4, max_epochs=30, patience=8)
    return _nn_proba(m, Xtr_nn), _nn_proba(m, Xte_nn)
ms_s4 = multiseed_eval(_s4_fit_fn)
all_results["S4_FrozenFinetune"] = {"loo": loo_s4, "ms": ms_s4, "train": train_m_s4, "prior_loo": loo_metrics(y_cftr, oof_s4, prior_shift=True)}
print(f"  LOO MCC={loo_s4['mcc']:.4f}  F1={loo_s4['f1']:.4f}  AUC={loo_s4['auc']:.4f}  boot-mean={loo_s4['boot8020']['mean']:.4f}")

# --- S5: BalancedBagging (MASTER uzerinde) ---
if HAS_IMBLEARN:
    print("S5: BalancedBagging (MASTER)...")
    bag = BalancedBaggingClassifier(
        estimator=_lgbm_classifier(), n_estimators=BAGGING_N,
        sampling_strategy="majority", random_state=SEED, n_jobs=-1)
    bag.fit(X_master_le, y_master)
    p_s5_train = bag.predict_proba(X_master_le)[:, 1]
    p_s5_cftr  = bag.predict_proba(X_cftr_le)[:, 1]
    thr_s5 = select_threshold_8020(y_master, p_s5_train)
    train_m_s5 = train_metrics_at(y_master, p_s5_train, thr_s5)
    loo_s5 = loo_metrics(y_cftr, p_s5_cftr)
    def _s5_fit_fn(Xtr_df, ytr, Xte_df, yte):
        b = BalancedBaggingClassifier(estimator=_lgbm_classifier(), n_estimators=BAGGING_N,
                                      sampling_strategy="majority", random_state=SEED)
        b.fit(X_master_le, y_master)
        Xte_le, _ = _le_encode_for_loo(Xte_df)
        return p_s5_train[:len(ytr)], b.predict_proba(Xte_le)[:, 1]
    ms_s5 = multiseed_eval(_s5_fit_fn)
    all_results["S5_BalancedBagging"] = {"loo": loo_s5, "ms": ms_s5, "train": train_m_s5, "prior_loo": loo_metrics(y_cftr, p_s5_cftr, prior_shift=True)}
    print(f"  LOO MCC={loo_s5['mcc']:.4f}  F1={loo_s5['f1']:.4f}  AUC={loo_s5['auc']:.4f}  boot-mean={loo_s5['boot8020']['mean']:.4f}")
else:
    all_results["S5_BalancedBagging"] = None; print("S5: ATLANDI (imblearn yok)")

# --- S6: PriorShift (S0 ciktisina post-hoc uygula) ---
print("S6: PriorShift (S0 + adjust_prior_shift)...")
p_s6_cftr = adjust_prior_shift(p_s0_cftr)
loo_s6 = loo_metrics(y_cftr, p_s6_cftr)  # prior_shift zaten uygulanmis
p_s6_train = adjust_prior_shift(p_s0_train)
thr_s6 = select_threshold_8020(y_master, p_s6_train)
train_m_s6 = train_metrics_at(y_master, p_s6_train, thr_s6)
# Multi-seed
def _s6_fit_fn(Xtr_df, ytr, Xte_df, yte):
    m = _lgbm_classifier(); m.fit(X_master_le, y_master)
    Xte_le, _ = _le_encode_for_loo(Xte_df)
    p_tr_raw = m.predict_proba(X_master_le)[:, 1][:len(ytr)]
    p_te_raw = m.predict_proba(Xte_le)[:, 1]
    return adjust_prior_shift(p_tr_raw), adjust_prior_shift(p_te_raw)
ms_s6 = multiseed_eval(_s6_fit_fn)
all_results["S6_PriorShift"] = {"loo": loo_s6, "ms": ms_s6, "train": train_m_s6, "prior_loo": loo_s6}
print(f"  LOO MCC={loo_s6['mcc']:.4f}  F1={loo_s6['f1']:.4f}  AUC={loo_s6['auc']:.4f}  boot-mean={loo_s6['boot8020']['mean']:.4f}")

print("\n=== Tum stratejiler tamamlandi ===")

=== LOO-CV basliyor (111 fold x her strateji) ===
S0: MASTER-only...
  LOO MCC=0.6271  F1=0.8982  AUC=0.9423  boot-mean=0.7281
S1: MASTER-core (625/625)...
  LOO MCC=0.6081  F1=0.8765  AUC=0.9122  boot-mean=0.7830
S2: SMOTE-CFTR...
  LOO MCC=0.5610  F1=0.9239  AUC=0.8376  boot-mean=0.4930
S3: ATLANDI (TabPFN v8 lisans/API uyumsuzlugu -- etkinlestirmek icin README'e bak)
S4: FrozenFinetune (SmallMLP, frozen body)...
  LOO MCC=0.2940  F1=0.5000  AUC=0.8566  boot-mean=0.3700
S5: BalancedBagging (MASTER)...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  LOO MCC=0.5717  F1=0.8553  AUC=0.9365  boot-mean=0.7535
S6: PriorShift (S0 + adjust_prior_shift)...
  LOO MCC=0.6552  F1=0.9186  AUC=0.9423  boot-mean=0.6915

=== Tum stratejiler tamamlandi ===


In [7]:
# Cell 7: Sonuc Derleme + Karsilastirma Tablosu

rows = []
NB16_REF = {"strategy": "NB16_ref (DNN, CI:[0-1])",
            "loo_mcc": "N/A", "loo_f1": "N/A", "loo_auc": "N/A",
            "boot_mean": 0.852, "boot_std": 0.267, "boot_lo": 0.00, "boot_hi": 1.00,
            "ms_f1_mean": "N/A", "ms_f1_std": "N/A", "ms_mcc_mean": "N/A",
            "train_f1": "N/A", "train_mcc": "N/A", "overfit_flag": "?"}
rows.append(NB16_REF)

for name, res in all_results.items():
    if res is None:
        rows.append({"strategy": name + " [ATLANDI]",
                     "loo_mcc": "N/A", "loo_f1": "N/A", "loo_auc": "N/A",
                     "boot_mean": "N/A", "boot_std": "N/A", "boot_lo": "N/A", "boot_hi": "N/A",
                     "ms_f1_mean": "N/A", "ms_f1_std": "N/A", "ms_mcc_mean": "N/A",
                     "train_f1": "N/A", "train_mcc": "N/A", "overfit_flag": "N/A"})
        continue
    loo = res["loo"]; ms = res["ms"]; tr = res["train"]
    boot = loo["boot8020"]
    overfit = (tr["train_f1"] - loo["f1"]) if isinstance(tr["train_f1"], float) else float("nan")
    rows.append({
        "strategy": name,
        "loo_mcc":   round(loo["mcc"], 4),
        "loo_f1":    round(loo["f1"], 4),
        "loo_auc":   round(loo["auc"], 4),
        "boot_mean": round(boot["mean"], 4),
        "boot_std":  round(boot["std"], 4),
        "boot_lo":   round(boot["lo"], 4),
        "boot_hi":   round(boot["hi"], 4),
        "ms_f1_mean":  round(ms["f1_mean"], 4),
        "ms_f1_std":   round(ms["f1_std"], 4),
        "ms_mcc_mean": round(ms["mcc_mean"], 4),
        "train_f1":  round(tr["train_f1"], 4) if isinstance(tr["train_f1"], float) else tr["train_f1"],
        "train_mcc": round(tr["train_mcc"], 4) if isinstance(tr["train_mcc"], float) else tr["train_mcc"],
        "overfit_flag": "YUKSEK" if overfit > 0.15 else ("ORTA" if overfit > 0.08 else "OK"),
    })

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(RESULTS_DIR, "cftr_results.csv"), index=False)

print("=== CFTR 7 Strateji Karsilastirmasi ===")
display_cols = ["strategy", "loo_mcc", "loo_f1", "loo_auc", "boot_mean", "boot_std",
                "ms_f1_mean", "ms_mcc_mean", "train_f1", "overfit_flag"]
print(results_df[display_cols].to_string(index=False))

# En iyi strateji LOO-MCC'ye gore
valid = results_df[results_df["loo_mcc"].apply(lambda x: isinstance(x, float))]
if len(valid) > 0:
    best = valid.loc[valid["loo_mcc"].idxmax()]
    print(f"\n>>> BIRINCIL METRIK LOO-MCC EN IYI: {best['strategy']} (MCC={best['loo_mcc']})")

print(f"\nSonuclar kaydedildi: {os.path.join(RESULTS_DIR, 'cftr_results.csv')}")

=== CFTR 7 Strateji Karsilastirmasi ===
                strategy loo_mcc  loo_f1 loo_auc boot_mean boot_std ms_f1_mean ms_mcc_mean train_f1 overfit_flag
NB16_ref (DNN, CI:[0-1])     N/A     N/A     N/A     0.852    0.267        N/A         N/A      N/A            ?
          S0_MASTER-only  0.6271  0.8982  0.9423    0.7281    0.107     0.9227      0.6573   0.9786         ORTA
          S1_MASTER-core  0.6081  0.8765  0.9122     0.783   0.0987     0.8759      0.5431   0.9846         ORTA
           S2_SMOTE-CFTR   0.561  0.9239  0.8376     0.493   0.0507     0.8967      0.2267      1.0           OK
     S3_TabPFN [ATLANDI]     N/A     N/A     N/A       N/A      N/A        N/A         N/A      N/A          N/A
       S4_FrozenFinetune   0.294     0.5  0.8566      0.37   0.2582     0.8072      0.4525      0.5           OK
      S5_BalancedBagging  0.5717  0.8553  0.9365    0.7535   0.1275     0.4977      0.2991   0.9347           OK
           S6_PriorShift  0.6552  0.9186  0.9423    0.69

In [8]:
# Cell 8: Gorsellestirmeler (Fig1-Fig5)

valid_res = {k: v for k, v in all_results.items() if v is not None}
strategies = list(valid_res.keys())
colors = ["#2ca02c", "#1f77b4", "#ff7f0e", "#9467bd", "#d62728", "#8c564b", "#e377c2"]
clr = {s: colors[i % len(colors)] for i, s in enumerate(strategies)}

# --- Fig1: LOO-MCC bar chart ---
fig, ax = plt.subplots(figsize=(12, 5))
mccs = [valid_res[s]["loo"]["mcc"] for s in strategies]
bars = ax.bar(strategies, mccs, color=[clr[s] for s in strategies], edgecolor="white", linewidth=0.8)
ax.set_ylim(-0.1, 1.0); ax.set_ylabel("LOO-CV MCC (birincil metrik)", fontsize=11)
ax.set_title("NB17 -- CFTR: 7 Strateji LOO-CV MCC Karsilastirmasi", fontweight="bold")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xticklabels(strategies, rotation=15, ha="right", fontsize=9)
for bar, val in zip(bars, mccs): ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                                          f"{val:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "fig1_loo_mcc.png"), dpi=110, bbox_inches="tight"); plt.show()

# --- Fig2: Multi-seed F1 dagilimi (violin benzeri: mean±std) ---
fig, ax = plt.subplots(figsize=(12, 5))
ms_means = [valid_res[s]["ms"]["f1_mean"] for s in strategies]
ms_stds  = [valid_res[s]["ms"]["f1_std"]  for s in strategies]
x = np.arange(len(strategies))
ax.bar(x, ms_means, yerr=ms_stds, color=[clr[s] for s in strategies],
       capsize=5, edgecolor="white", linewidth=0.8)
ax.set_ylim(0, 1.1); ax.set_ylabel("Multi-seed F1 (mean ± std, 20 seed)", fontsize=11)
ax.set_title("NB17 -- CFTR: Multi-seed 50/50 F1 (Tek-split yanilsamasini kirar)", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(strategies, rotation=15, ha="right", fontsize=9)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "fig2_multiseed_f1.png"), dpi=110, bbox_inches="tight"); plt.show()

# --- Fig3: Prior-shift duzeltmesi etkisi (S0: raw vs adjusted olasilik histogrami) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
p_raw = p_s0_cftr; p_adj = adjust_prior_shift(p_s0_cftr)
for ax, p, title in zip(axes, [p_raw, p_adj], ["S0 raw olasilik", "S0 prior-shift duzeltmesi (pi_test=0.20)"]):
    for lbl, color in [(0, "#2ca02c"), (1, "#d62728")]:
        ax.hist(p[y_cftr == lbl], bins=25, alpha=0.6, color=color,
                label="Benign" if lbl == 0 else "Pathogenic", density=True)
    ax.set_xlabel("Tahmin olasiligi"); ax.set_ylabel("Yogunluk")
    ax.set_title(title); ax.legend(fontsize=9); ax.axvline(0.5, color="gray", linestyle="--")
fig.suptitle("NB17 -- Prior Shift Duzeltmesi: Olasilik Dagilimi", fontweight="bold")
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "fig3_prior_shift.png"), dpi=110, bbox_inches="tight"); plt.show()

# --- Fig4: Confusion matrix (en iyi LOO-MCC stratejisi) ---
from sklearn.metrics import ConfusionMatrixDisplay
best_key = max(valid_res, key=lambda k: valid_res[k]["loo"]["mcc"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, key, ttl in zip(axes, [best_key, "S6_PriorShift"], [f"En iyi: {best_key}", "S6: PriorShift"]):
    if key not in valid_res: ax.set_visible(False); continue
    cm = confusion_matrix(valid_res[key]["loo"]["y_true"],
                          valid_res[key]["loo"]["y_pred"], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Benign", "Pathogenic"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(ttl, fontsize=9)
fig.suptitle("NB17 -- CFTR Confusion Matrix (LOO-CV)", fontweight="bold")
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "fig4_confusion.png"), dpi=110, bbox_inches="tight"); plt.show()

# --- Fig5: Train vs Test F1/MCC (overfit kontrolu) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_f1s  = [valid_res[s]["train"]["train_f1"] for s in strategies]
test_f1s   = [valid_res[s]["loo"]["f1"] for s in strategies]
train_mccs = [valid_res[s]["train"]["train_mcc"] for s in strategies]
test_mccs  = [valid_res[s]["loo"]["mcc"] for s in strategies]
x = np.arange(len(strategies)); w = 0.35
for ax, tr_vals, te_vals, ylabel in zip(
        axes,
        [train_f1s, train_mccs],
        [test_f1s,  test_mccs],
        ["F1", "MCC"]):
    bars_tr = ax.bar(x - w/2, tr_vals, w, label="Train", color="#1f77b4", alpha=0.8)
    bars_te = ax.bar(x + w/2, te_vals, w, label="Test (LOO-CV)", color="#ff7f0e", alpha=0.8)
    # Overfit uyarisi (fark > 0.15)
    for i, (tr, te) in enumerate(zip(tr_vals, te_vals)):
        if isinstance(tr, float) and isinstance(te, float) and (tr - te) > 0.15:
            ax.annotate("OVERFIT", xy=(x[i], max(tr, te) + 0.02),
                        ha="center", color="red", fontsize=7, fontweight="bold")
    ax.set_ylim(-0.1, 1.15); ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"Train vs Test {ylabel} (overfit kontrolu)", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(strategies, rotation=15, ha="right", fontsize=8)
    ax.legend(fontsize=9); ax.axhline(0, color="gray", linestyle="--", alpha=0.4)
fig.suptitle("NB17 -- CFTR: Train vs Test Karsilastirmasi (NB15 v1 Tuzagindan Ders)", fontweight="bold")
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "fig5_train_vs_test.png"), dpi=110, bbox_inches="tight"); plt.show()

print("5 gorsel kaydedildi.")

5 gorsel kaydedildi.


In [9]:
# Cell 9: Ozet & Tartisma

valid = results_df[results_df["loo_mcc"].apply(lambda x: isinstance(x, float))]
if len(valid) > 0:
    best_row = valid.loc[valid["loo_mcc"].idxmax()]
    print(f"""\n{'='*70}
NB17 CFTR REFINEMENT -- OZET
{'='*70}

BIRINCIL METRIK (LOO-CV MCC):
  En iyi strateji : {best_row['strategy']}
  LOO-CV MCC      : {best_row['loo_mcc']}
  LOO-CV F1       : {best_row['loo_f1']}
  LOO-CV AUC      : {best_row['loo_auc']}
  Boot %80/20 F1  : {best_row['boot_mean']} ± {best_row['boot_std']}
  Multi-seed F1   : {best_row['ms_f1_mean']} ± {best_row['ms_f1_std']}
  Overfit durumu  : {best_row['overfit_flag']}

NB16 referans (DNN, CI=[0.00-1.00]): boot_mean=0.852 (guvenilmez!)
LOO-CV ile degerlendirme: NB16'dan daha anlamli (tum 21 benign kullanildi)

{'='*70}
METODOLOJIK BULGULAR:
- Bootstrap CI [0-1] problemi LOO-CV ile giderildi -> anlamli MCC metrigi
- Prior shift duzeltmesi (S6) S0'i nasil degeristirdi: {results_df[results_df.strategy=='S6_PriorShift']['loo_mcc'].values}
- Overfit durumu (train-test farki > 0.15): {results_df[results_df.overfit_flag=='YUKSEK']['strategy'].tolist()}
{'='*70}""")

print("\nFINAL TESLIM ONERISI:")
print("  CFTR icin: LOO-MCC en yuksek stratejiyi kullan.")
print("  Precision=1.0 olanlari tercih et (FP yok -> benign tarafi guvenli).")
print("  PAH'a transfer edilebilir bulgular: prior-shift duzeltmesi, BalancedBagging.")


NB17 CFTR REFINEMENT -- OZET

BIRINCIL METRIK (LOO-CV MCC):
  En iyi strateji : S6_PriorShift
  LOO-CV MCC      : 0.6552
  LOO-CV F1       : 0.9186
  LOO-CV AUC      : 0.9423
  Boot %80/20 F1  : 0.6915 ± 0.0877
  Multi-seed F1   : 0.9227 ± 0.0167
  Overfit durumu  : OK

NB16 referans (DNN, CI=[0.00-1.00]): boot_mean=0.852 (guvenilmez!)
LOO-CV ile degerlendirme: NB16'dan daha anlamli (tum 21 benign kullanildi)

METODOLOJIK BULGULAR:
- Bootstrap CI [0-1] problemi LOO-CV ile giderildi -> anlamli MCC metrigi
- Prior shift duzeltmesi (S6) S0'i nasil degeristirdi: [0.6552]
- Overfit durumu (train-test farki > 0.15): []

FINAL TESLIM ONERISI:
  CFTR icin: LOO-MCC en yuksek stratejiyi kullan.
  Precision=1.0 olanlari tercih et (FP yok -> benign tarafi guvenli).
  PAH'a transfer edilebilir bulgular: prior-shift duzeltmesi, BalancedBagging.


In [10]:
# Cell 10: PDF Rapor
from fpdf import FPDF
from PIL import Image

class NB17Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "", 9); self.set_text_color(90, 90, 90)
        self.cell(0, 8, "NB17 - CFTR Refinement | TEKNOFEST Genetik Varyant", align="C", ln=True)
        self.set_text_color(0, 0, 0)
    def footer(self):
        self.set_y(-15); self.set_font("Helvetica", "", 8)
        self.set_text_color(120, 120, 120); self.cell(0, 10, f"Sayfa {self.page_no()}", align="C")
    def section(self, t):
        self.ln(2); self.set_fill_color(31, 119, 180); self.set_text_color(255, 255, 255)
        self.set_font("Helvetica", "B", 12)
        self.cell(0, 9, f"  {t}", ln=True, fill=True); self.set_text_color(0, 0, 0); self.ln(2)
    def body(self, t): self.set_font("Helvetica", "", 10); self.multi_cell(0, 5.5, t); self.ln(1)
    def table(self, h, d, w):
        self.set_font("Helvetica", "B", 8); self.set_fill_color(70, 130, 180); self.set_text_color(255, 255, 255)
        for x, wi in zip(h, w): self.cell(wi, 7, str(x), border=1, align="C", fill=True)
        self.ln(); self.set_text_color(0, 0, 0); self.set_font("Helvetica", "", 8); fill = False
        for row in d:
            self.set_fill_color(235, 240, 248)
            for v, wi in zip(row, w): self.cell(wi, 6, str(v), border=1, align="C", fill=fill)
            self.ln(); fill = not fill
    def uw(self): return self.w - self.l_margin - self.r_margin
    def fit_image(self, p, max_h=None):
        try:
            iw, ih = Image.open(p).size; w = self.uw(); h = w * ih / iw
            if max_h and h > max_h: h = max_h; w = h * iw / ih
            self.image(p, w=w, h=h)
        except Exception: self.body(f"[Gorsel yuklenemedi: {p}]")

pdf = NB17Report(); pdf.set_auto_page_break(auto=True, margin=18)

# --- Kapak ---
pdf.add_page(); pdf.set_font("Helvetica", "B", 18); pdf.ln(6)
pdf.cell(0, 12, "NB17: CFTR Panel Refinement - 7 Strateji", align="C", ln=True)
pdf.set_font("Helvetica", "", 12)
pdf.cell(0, 8, "TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant", align="C", ln=True)
pdf.cell(0, 8, f"Tarih: {datetime.now():%Y-%m-%d %H:%M}", align="C", ln=True)

# --- 1. Problem ---
pdf.section("1. Problem: Kucuk N, Buyuk Dengesizlik")
pdf.body("CFTR paneli n=21 benign ile degerlendirilemiyor. NB15/NB16'da bootstrap CI=[0.00-1.00]. "
         "Bu notebook 3 katmanli cozum uygular: (1) LOO-CV+MCC degerlendirme, "
         "(2) prior-shift kalibrasyon, (3) 7 strateji karsilastirmasi.")

# --- 2. Metodoloji ---
pdf.section("2. Metodoloji")
pdf.body("LOO-CV: 111 fold, tum 21 benign kullanilir. Birincil metrik: MCC (Chicco & Jurman 2020).\n"
         "Prior shift: Saerens et al. (2002) formulu, pi_train=0.73 -> pi_test=0.20.\n"
         "Multi-seed: 20 farkli seed x 50/50 split -> F1 dagilimi.\n"
         "Train metrigi: overfit kontrolu (gap > 0.15 = YUKSEK, > 0.08 = ORTA).\n"
         "S0/S1/S5/S6: MASTER egitimi -> CFTR testi. S2: SMOTE. S3: TabPFN. S4: FrozenFinetune.")

# --- 3. Sonuclar Tablosu ---
pdf.section("3. Sonuclar: 7 Strateji Karsilastirmasi")
hdr = ["Strateji", "LOO-MCC", "LOO-F1", "LOO-AUC", "Boot-mean", "Boot-std", "MS-F1", "Train-F1", "Overfit"]
wts = [40, 20, 18, 18, 22, 18, 18, 20, 18]
data_rows = []
for _, row in results_df.iterrows():
    data_rows.append([str(row["strategy"])[:25], str(row["loo_mcc"]), str(row["loo_f1"]),
                      str(row["loo_auc"]), str(row["boot_mean"]), str(row["boot_std"]),
                      str(row["ms_f1_mean"]), str(row["train_f1"]), str(row["overfit_flag"])])
pdf.table(hdr, data_rows, wts)

# --- 4. Gorsel ---
pdf.section("4. Gorseller")
for figname, caption in [
    ("fig1_loo_mcc.png", "Fig1: LOO-CV MCC -- 7 Strateji"),
    ("fig2_multiseed_f1.png", "Fig2: Multi-seed F1 (mean+-std, 20 seed)"),
    ("fig3_prior_shift.png", "Fig3: Prior-Shift Duzeltmesi Etkisi (S0 raw vs adjusted)"),
    ("fig4_confusion.png", "Fig4: Confusion Matrix (LOO-CV, en iyi + S6)"),
    ("fig5_train_vs_test.png", "Fig5: Train vs Test F1/MCC (Overfit Kontrolu)"),
]:
    fp = os.path.join(RESULTS_DIR, figname)
    if os.path.exists(fp):
        pdf.ln(3); pdf.set_font("Helvetica", "I", 9); pdf.cell(0, 6, caption, ln=True)
        pdf.fit_image(fp, max_h=80)

# --- 5. Oneri ---
pdf.section("5. Final Onerim")
valid_r = results_df[results_df["loo_mcc"].apply(lambda x: isinstance(x, (int, float)))]
if len(valid_r) > 0:
    best_s = valid_r.loc[valid_r["loo_mcc"].idxmax()]
    pdf.body(f"En iyi strateji (LOO-MCC): {best_s['strategy']} -> MCC={best_s['loo_mcc']}\n"
             "CFTR icin final teslim oncelik sirasi: (1) LOO-MCC en yuksek, "
             "(2) Precision=1.0 (FP yok), (3) Overfit=OK.\n"
             "PAH'a transfer edilebilir: prior-shift duzeltmesi + BalancedBagging.")

out_pdf = os.path.join(REPORTS_DIR, "NB17_cftr_refinement_report.pdf")
pdf.output(out_pdf)
print(f"PDF rapor kaydedildi: {out_pdf}")

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB17_cftr_refinement_report.pdf


In [ ]:
# Cell 11: Regularizasyon Sweep -- Tasarim + Yardimci Fonksiyon
#
# Amac: Cell 7'deki "overfit" bayragi = train_f1 - loo_f1, MASTER-egitimli
# stratejiler (S0/S1/S5/S6) icin train_f1 MASTER'da, loo_f1 CFTR'de olculur --
# bu fark GERCEK ezberleme (memorization) ile MASTER->CFTR DAGILIM KAYMASI'ni
# (domain shift) birbirine karistirir. Bu hucreler iki etkiyi ayristirir ve
# LightGBM'i regularize etmenin CFTR transferine bedavaya mi yoksa pahaliya mi
# geldigini test eder.
#
# Iki temiz 1-D eksen (CLAUDE.md: tek seferde bir sey degistir):
#   Eksen A -- min_child_samples in [5,10,20,40,80,160,320]  (num_leaves=31 sabit)
#   Eksen B -- num_leaves in [63,31,15,7,3]  (min_child_samples=20 sabit)
# (min_child_samples=20 / num_leaves=31 = mevcut S0 baseline, her iki eksende
#  de yer alir -- egrileri ankraj eder.)

def _lgbm_reg(**ov):
    base = {**LGBM_FIXED, "n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31}
    base.update(ov)
    return lgb.LGBMClassifier(**base)

AXIS_A_VALUES = [5, 10, 20, 40, 80, 160, 320]   # min_child_samples sweep @ num_leaves=31
AXIS_B_VALUES = [63, 31, 15, 7, 3]               # num_leaves sweep @ min_child_samples=20

BASELINE_MCS = 20
BASELINE_NL  = 31

sweep_configs = []
for v in AXIS_A_VALUES:
    sweep_configs.append(("min_child_samples", v, {"min_child_samples": v, "num_leaves": BASELINE_NL}))
for v in AXIS_B_VALUES:
    sweep_configs.append(("num_leaves", v, {"min_child_samples": BASELINE_MCS, "num_leaves": v}))

print(f"Sweep konfigurasyon sayisi: {len(sweep_configs)}")

def select_threshold_8020_robust(y, prob, n_resample=N_BOOT):
    """N farkli %80/20 yeniden-ornekleme uzerinde ORTALAMA pathogenic-F1'i maksimize
    eden esik. Tek-resample gurultusunu giderir -> master_cv_f1/memorization_gap guvenilir."""
    thr_grid = np.arange(0.05, 0.95, 0.01)
    acc = np.zeros(len(thr_grid))
    for s in range(n_resample):
        rng = np.random.RandomState(BOOT_SEED + s)
        yb, pb = _resample_8020(y, prob, rng)
        for j, thr in enumerate(thr_grid):
            acc[j] += _f1_pos(yb, (pb >= thr).astype(int))
    return float(thr_grid[int(np.argmax(acc))])

In [ ]:
# Cell 12: Regularizasyon Sweep -- Calistir
#
# Her config icin:
#   - train_f1            : tum MASTER'da fit, select_threshold_8020 ile esik, train_metrics_at
#   - master_cv_f1/mcc     : MASTER'da 5-fold StratifiedKFold OOF (AYNI parametreler) -- temiz
#                            ESDEGER-DAGILIM referansi
#   - memorization_gap     = train_f1 - master_cv_f1   (GERCEK overfit)
#   - CFTR transferi (S6 tarzi, prior-shift ON)        -> loo_mcc/f1/auc, boot8020
#   - transfer_gap          = master_cv_f1 - loo_f1    (MASTER->CFTR domain shift)
#   - multi-seed 50/50 (S6 _s6_fit_fn deseni, swept params) -> ms_f1_mean/std

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

sweep_rows = []
for axis, value, ov in sweep_configs:
    print(f"[{axis}={value}] egitiliyor...")
    params = ov

    # 1) Tum MASTER uzerinde fit -> train_f1
    m_full = _lgbm_reg(**params)
    m_full.fit(X_master_le, y_master)
    p_master_train = m_full.predict_proba(X_master_le)[:, 1]
    thr_train = select_threshold_8020_robust(y_master, p_master_train)
    train_m = train_metrics_at(y_master, p_master_train, thr_train)
    train_f1 = train_m["train_f1"]

    # 2) 5-fold StratifiedKFold OOF on MASTER -> master_cv_f1 / master_cv_mcc
    oof_master = np.zeros(len(y_master), dtype=float)
    for tr_idx, va_idx in skf.split(X_master_le, y_master):
        m_cv = _lgbm_reg(**params)
        m_cv.fit(X_master_le.iloc[tr_idx], y_master[tr_idx])
        oof_master[va_idx] = m_cv.predict_proba(X_master_le.iloc[va_idx])[:, 1]
    thr_cv = select_threshold_8020_robust(y_master, oof_master)
    yp_cv = (oof_master >= thr_cv).astype(int)
    master_cv_f1 = _f1_pos(y_master, yp_cv)
    master_cv_mcc = matthews_corrcoef(y_master, yp_cv)

    memorization_gap = train_f1 - master_cv_f1

    # 3) CFTR transfer (S6-style, prior-shift ON)
    p_cftr = m_full.predict_proba(X_cftr_le)[:, 1]
    res = loo_metrics(y_cftr, p_cftr, prior_shift=True)
    loo_mcc = res["mcc"]; loo_f1 = res["f1"]; loo_auc = res["auc"]
    boot_mean = res["boot8020"]["mean"]; boot_std = res["boot8020"]["std"]

    transfer_gap = master_cv_f1 - loo_f1

    # 4) Multi-seed (S6 _s6_fit_fn pattern, swept params)
    def _fit_fn(Xtr_df, ytr, Xte_df, yte, _ov=params):
        m = _lgbm_reg(**_ov); m.fit(X_master_le, y_master)
        Xte_le, _ = _le_encode_for_loo(Xte_df)
        p_tr_raw = m.predict_proba(X_master_le)[:, 1][:len(ytr)]
        p_te_raw = m.predict_proba(Xte_le)[:, 1]
        return adjust_prior_shift(p_tr_raw), adjust_prior_shift(p_te_raw)
    ms = multiseed_eval(_fit_fn)
    ms_f1_mean = ms["f1_mean"]; ms_f1_std = ms["f1_std"]

    sweep_rows.append({
        "axis": axis, "value": value,
        "train_f1": round(train_f1, 4),
        "master_cv_f1": round(master_cv_f1, 4),
        "master_cv_mcc": round(master_cv_mcc, 4),
        "memorization_gap": round(memorization_gap, 4),
        "loo_mcc": round(loo_mcc, 4),
        "loo_f1": round(loo_f1, 4),
        "loo_auc": round(loo_auc, 4),
        "boot_mean": round(boot_mean, 4),
        "boot_std": round(boot_std, 4),
        "transfer_gap": round(transfer_gap, 4),
        "ms_f1_mean": round(ms_f1_mean, 4),
        "ms_f1_std": round(ms_f1_std, 4),
    })
    print(f"  train_f1={train_f1:.4f}  master_cv_f1={master_cv_f1:.4f}  "
          f"loo_mcc={loo_mcc:.4f}  loo_f1={loo_f1:.4f}  "
          f"mem_gap={memorization_gap:.4f}  transfer_gap={transfer_gap:.4f}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_csv = os.path.join(RESULTS_DIR, "reg_sweep.csv")
sweep_df.to_csv(sweep_csv, index=False)
print(f"\nSweep sonuclari kaydedildi: {sweep_csv}")
print(sweep_df.to_string(index=False))

In [ ]:
# Cell 13: Figur -- fig6_reg_sweep.png
#
# Iki alt-grafik (her eksen icin biri): x = knob degeri (artan regularizasyon
# yonunde siralanmis), sol y = CFTR loo_mcc (cizgi+marker, birincil metrik) +
# ms_f1_mean (kesikli, std bandi); sag y (twinx) = memorization_gap ve
# transfer_gap (kesikli cizgiler). Baseline config (min_child_samples=20 /
# num_leaves=31) dikey cizgiyle isaretlenir.

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axis_specs = [
    ("min_child_samples", AXIS_A_VALUES, BASELINE_MCS),
    ("num_leaves", AXIS_B_VALUES, BASELINE_NL),
]

for ax, (axis_name, values, baseline_val) in zip(axes, axis_specs):
    sub = sweep_df[sweep_df["axis"] == axis_name].sort_values("value")
    x = sub["value"].values

    # Sol y-ekseni: loo_mcc + ms_f1_mean bandi
    l1, = ax.plot(x, sub["loo_mcc"], "o-", color="#1f77b4", label="CFTR LOO-MCC (prior-shift)", linewidth=2)
    l2, = ax.plot(x, sub["ms_f1_mean"], "s--", color="#2ca02c", label="Multi-seed F1 (mean)", linewidth=1.5)
    ax.fill_between(x, sub["ms_f1_mean"] - sub["ms_f1_std"], sub["ms_f1_mean"] + sub["ms_f1_std"],
                    color="#2ca02c", alpha=0.15)
    ax.set_xlabel(f"{axis_name} (artan regularizasyon ->)", fontsize=10)
    ax.set_ylabel("CFTR Skoru (MCC / F1)", fontsize=10)
    ax.set_ylim(-0.2, 1.05)

    # Sag y-ekseni: gap'ler
    axr = ax.twinx()
    l3, = axr.plot(x, sub["memorization_gap"], "^:", color="#d62728", label="memorization_gap (train-mastercv)", linewidth=1.5)
    l4, = axr.plot(x, sub["transfer_gap"], "v:", color="#9467bd", label="transfer_gap (mastercv-loo)", linewidth=1.5)
    axr.set_ylabel("Gap (F1 farki)", fontsize=10)
    axr.axhline(0, color="gray", linestyle="-", alpha=0.3)

    # Baseline isareti
    ax.axvline(baseline_val, color="black", linestyle="--", alpha=0.5, linewidth=1)
    ax.text(baseline_val, ax.get_ylim()[1] * 0.97, " baseline\n (S0)", fontsize=8,
            ha="left" if axis_name == "min_child_samples" else "right", va="top")

    if axis_name == "num_leaves":
        ax.invert_xaxis()  # artan regularizasyon = azalan num_leaves

    ax.set_title(axis_name, fontweight="bold")
    ax.grid(alpha=0.2)

    lines = [l1, l2, l3, l4]
    ax.legend(lines, [ln.get_label() for ln in lines], fontsize=8, loc="lower left")

fig.suptitle("NB17 -- CFTR: Regularizasyon Gucu vs Transfer Basarisi\n"
              "(Overfit'i cozmek basari kaybettiriyor mu?)", fontweight="bold")
plt.tight_layout()
fig6_path = os.path.join(RESULTS_DIR, "fig6_reg_sweep.png")
fig.savefig(fig6_path, dpi=110, bbox_inches="tight")
plt.show()
print(f"Figur kaydedildi: {fig6_path}")

In [ ]:
# Cell 14: Verdict -- Regularizasyon CFTR Transferine Bedava mi, Pahali mi?
#
# Baseline S0 (min_child_samples=20, num_leaves=31) loo_mcc'sini sweep'teki
# en iyi (ve daha-regularize) konfigurasyonlarla kiyaslar; memorization_gap
# vs transfer_gap ayrismasini raporlar.
#
# [UYARI] n=21 benign -- sadece |loo_mcc farki| > 0.05 olan kaymalara gore
# aksiyon al; 0.02-0.05 'oneri' niteliginde, < 0.02 gurultu (CFTR DEGERLENDIRME
# GUVENI YOK bolumu, CLAUDE.md).

baseline_row = sweep_df[(sweep_df["axis"] == "min_child_samples") & (sweep_df["value"] == BASELINE_MCS)]
if len(baseline_row) == 0:
    baseline_row = sweep_df[(sweep_df["axis"] == "num_leaves") & (sweep_df["value"] == BASELINE_NL)]
baseline_row = baseline_row.iloc[0]
baseline_loo_mcc = baseline_row["loo_mcc"]
baseline_mem_gap = baseline_row["memorization_gap"]
baseline_transfer_gap = baseline_row["transfer_gap"]

# "Daha regularize" konfigurasyonlar: min_child_samples > baseline VEYA num_leaves < baseline
more_reg_mask = (
    ((sweep_df["axis"] == "min_child_samples") & (sweep_df["value"] > BASELINE_MCS)) |
    ((sweep_df["axis"] == "num_leaves") & (sweep_df["value"] < BASELINE_NL))
)
more_reg_df = sweep_df[more_reg_mask]

best_idx = sweep_df["loo_mcc"].idxmax()
best_row = sweep_df.loc[best_idx]
best_loo_mcc = best_row["loo_mcc"]

best_more_reg_idx = more_reg_df["loo_mcc"].idxmax() if len(more_reg_df) > 0 else None
best_more_reg_row = sweep_df.loc[best_more_reg_idx] if best_more_reg_idx is not None else None

print("="*70)
print("NB17 -- REGULARIZASYON SWEEP VERDICT")
print("="*70)
print(f"\nBaseline (S0: min_child_samples={BASELINE_MCS}, num_leaves={BASELINE_NL}):")
print(f"  loo_mcc           = {baseline_loo_mcc:.4f}")
print(f"  memorization_gap  = {baseline_mem_gap:.4f}  (train_f1 - master_cv_f1, GERCEK overfit)")
print(f"  transfer_gap      = {baseline_transfer_gap:.4f}  (master_cv_f1 - loo_f1, MASTER->CFTR domain shift)")
if abs(baseline_mem_gap) >= abs(baseline_transfer_gap):
    print(f"  -> Baseline'da MEMORIZATION_GAP daha buyuk/esit ({baseline_mem_gap:.4f} vs {baseline_transfer_gap:.4f})")
else:
    print(f"  -> Baseline'da TRANSFER_GAP daha buyuk ({baseline_transfer_gap:.4f} vs {baseline_mem_gap:.4f}) "
          f"-> 'overfit' gap'inin cogu domain-shift kaynakli, gercek memorization kucuk.")

print(f"\nSweep genelinde en iyi loo_mcc: {best_loo_mcc:.4f}  ({best_row['axis']}={best_row['value']})")
if best_more_reg_row is not None:
    print(f"Daha regularize konfigurasyonlar icinde en iyi loo_mcc: "
          f"{best_more_reg_row['loo_mcc']:.4f}  ({best_more_reg_row['axis']}={best_more_reg_row['value']}, "
          f"memorization_gap={best_more_reg_row['memorization_gap']:.4f})")

print(f"\n[UYARI] n=21 benign -- yalniz |loo_mcc farki| > 0.05 olan kaymalara gore aksiyon al; "
      f"0.02-0.05 'oneri' niteliginde, < 0.02 gurultu.")

verdict = None
if best_more_reg_row is not None:
    delta = best_more_reg_row["loo_mcc"] - baseline_loo_mcc
    if delta > 0.02 and best_more_reg_row["memorization_gap"] < baseline_mem_gap:
        verdict = ("A", f"{best_more_reg_row['axis']}={best_more_reg_row['value']}")
    else:
        verdict = ("B/C", None)
else:
    verdict = ("B/C", None)

print("\n" + "-"*70)
if verdict[0] == "A":
    print(f"VERDICT A: {verdict[1]} hem overfit'i azaltiyor hem CFTR LOO-MCC'yi artiriyor "
          f"-> bedava kazanc, S6'yi bu config ile guncelle.")
else:
    print("VERDICT B/C: Regularizasyon CFTR LOO-MCC'yi artirmiyor -> mevcut overfit 'gap'i "
          "buyuk olcude domain-shift; sikilastirmak gercek basari kaybettirir. "
          "S6 mevcut haliyle birak.")
print("-"*70)